# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Mathematical Modeling & Quantum Mechanics

---
*Note: This notebook serves as a numerical solver for Partial Differential Equations (PDEs). It applies the Crank-Nicolson method to the time-dependent Schrödinger equation to model quantum tunneling. Operationally, it abstracts the physical phenomenon into the recurring inversion of sparse tridiagonal matrices over a discretized spatial grid, propagating Gaussian wave functions.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

# Simplified Quantum Constants (hbar = 2m = 1)
dt = 0.005
time_steps = 400

### 1. Grid Ingestion and Potential Barrier $V(x)$
We load the pre-calculated spatial grid and barrier profile, defining the one-dimensional topology on which our particle (wave packet) will interact.

In [ ]:
df_pot = pd.read_csv('../data/quantum_barrier_profile.csv')
x = df_pot['x'].values
V = df_pot['V'].values

N = len(x)
dx = x[1] - x[0]

print(f"[*] Mesh discretization: {N} spatial nodes. Differential dx = {dx:.4f}")

### 2. Initial Condition: The Gaussian Wave Packet $\Psi(x,0)$
We initialize the particle as a probability density by injecting intrinsic momentum ($k_0$) pointing towards the limiting barrier.

In [ ]:
x0 = -2.0      # Initial position
sigma = 0.25   # Dispersion (Spatial uncertainty)
k0 = 15.0      # Momentum (Group velocity)

# Complex Field Construction
psi_0 = np.exp(-0.5 * ((x - x0)/sigma)**2) * np.exp(1j * k0 * x)

# Unitary Normalization (The sum of probabilities must be exactly 1)
norm = np.sum(np.abs(psi_0)**2) * dx
psi = psi_0 / np.sqrt(norm)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, np.abs(psi)**2, color='cyan', label='Initial Density $|\\Psi(x,0)|^2$')
ax.fill_between(x, V / np.max(V) * np.max(np.abs(psi)**2), color='red', alpha=0.3, label='Potential Barrier V(x)')
ax.set_title("Boundary Conditions and Initial Packet", fontsize=14, pad=10)
ax.legend(loc="upper left")
plt.show()

### 3. Tridiagonal Sparse Matrices (Crank-Nicolson)
By discretizing the Schrödinger Equation:
$i\frac{\partial \Psi}{\partial t} = \left(-\frac{\partial^2}{\partial x^2} + V(x)\right)\Psi$

We obtain the operator of the Hamiltonian system $(H)$. The unconditionally stable Crank-Nicolson method is formalized by isolating future time iterations $\Psi^{n+1}$:
$\left(I + \frac{i \Delta t}{2} H \right) \Psi^{n+1} = \left(I - \frac{i \Delta t}{2} H \right) \Psi^{n}$

In [ ]:
# Laplacian Coefficient
alpha = 1.0 / (dx**2)

# Main Diagonal: (2/dx^2 + V(x))
d0 = 2.0 * alpha + V
# Secondary Diagonals: (-1/dx^2)
d1 = -alpha * np.ones(N)
dm1 = -alpha * np.ones(N)

# Construct highly efficient sparse matrix (SciPy sparse)
H = sparse.diags([dm1, d0, d1], offsets=[-1, 0, 1], shape=(N, N), format='csc')

# Construct Temporal Evolution Matrices (M_L * Psi_new = M_R * Psi_old)
I = sparse.eye(N, format='csc')
coef = (1j * dt) / 2.0

M_L = I + coef * H
M_R = I - coef * H

# Factorization and Direct Linear Solver
solver = spla.factorized(M_L)
print("[*] Factorized Vector Matrix. Direct linear solver initialized.")

### 4. Time Propagation and Transmission Coefficient Calculation
We advance loop by loop solving $A x = b$. The part of the wave that manages to cross the numerical width of $V_0=50$ will experimentally determine the occurrence of the Quantum Tunneling Phenomenon.

In [ ]:
density_history = []

# Crank-Nicolson propagation cycle
for step in range(time_steps):
    # Mat-vec multiplication for independent term b
    b = M_R.dot(psi)
    # Mathematical inversion to solve Psi^{n+1}
    psi = solver(b)
    
    if step % 80 == 0 or step == time_steps - 1:
        density_history.append(np.abs(psi)**2)
        
# Integral Quantification
# Total probability density must still be 1 (conservation)
total_prob = np.sum(np.abs(psi)**2) * dx

# Transmitted Probability (Integrating space after X=1.0)
idx_trans = np.where(x > 1.0)[0]
trans_prob = np.sum(np.abs(psi[idx_trans])**2) * dx

print(f"[+] Packet Conservation (Norm): {total_prob:.4f}")
print(f"[+] Quantum Transmission Coefficient (Tunneling Probability): {trans_prob*100:.3f} %")

### Final Visualization
We show how the tridiagonal sparse matrix has "cut" and "bounced" the mathematical wave front. A statistically low fraction of the vector reached across the solid mass due to the internal exponential decay in the resolving matrix.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.fill_between(x, V / np.max(V) * np.max(density_history[0]), color='red', alpha=0.1, label='Obstacle Matrix (V_x)')

colors = ['cyan', 'dodgerblue', 'blueviolet', 'magenta', 'lime', 'yellow']

for i, p_dens in enumerate(density_history):
    ax.plot(x, p_dens, color=colors[i % len(colors)], label=f'Time Step {i*80}')

ax.set_title("Quantum Tunneling: Density Signal Scattering and Transmission", fontsize=15, pad=15)
ax.set_xlabel("Spatial Mesh X")
ax.set_ylabel("Probability Density $|\\Psi(x,t)|^2$")
ax.set_xlim(-4, 4)
ax.legend(loc="upper right", fontsize=9)
plt.grid(alpha=0.2)
plt.show()